In [2]:
# STEP 1: Install required libraries
!pip install transformers datasets accelerate -q


 STEP 2: Import necessary modules

In [3]:
import torch
from datasets import load_dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)


STEP 3: Load the Dataset (WritingPrompts)

In [4]:
# This dataset contains human-written creative stories
dataset = load_dataset("euclaise/writingprompts", split="train")

# Display one sample to understand dataset structure
dataset[0]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/837 [00:00<?, ?B/s]

data/train-00000-of-00002-105e07cb0d1994(…):   0%|          | 0.00/272M [00:00<?, ?B/s]

data/train-00001-of-00002-4fdb982c110564(…):   0%|          | 0.00/272M [00:00<?, ?B/s]

data/test-00000-of-00001-16503b0c26ed00c(…):   0%|          | 0.00/30.0M [00:00<?, ?B/s]

data/validation-00000-of-00001-137b93e1e(…):   0%|          | 0.00/30.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/272600 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/15138 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15620 [00:00<?, ? examples/s]

{'prompt': "[ WP ] You 've finally managed to discover the secret to immortality . Suddenly , Death appears before you , hands you a business card , and says , `` When you realize living forever sucks , call this number , I 've got a job offer for you . ''\n",
 'story': "So many times have I walked on ruins, the remainings of places that I loved and got used to.. At first I was scared, each time I could feel my city, my current generation collapse, break into the black hole that thrives within it, I could feel humanity, the way I'm able to feel my body.. After a few hundred years, the pattern became obvious, no longer the war and damage that would devastate me over and over again in the far past was effecting me so dominantly. \n It's funny, but I felt as if after gaining what I desired so long, what I have lived for my entire life, only then, when I achieved immortality I started truly aging. \n \n 5 world wars have passed, and now they feel like a simple sickeness that would pass by 

STEP 4: Dataset Cleaning (Important for Model Quality)

In [5]:
# Keep only the story text
# Removing prompts and metadata avoids confusing the model
dataset = dataset.map(lambda x: {"text": x["story"]})
dataset = dataset.remove_columns(["prompt", "story"])


Map:   0%|          | 0/272600 [00:00<?, ? examples/s]

STEP 5: Load Pre-trained GPT-2 Model and Tokenizer

In [6]:
# Load GPT-2 tokenizer (text to tokens)
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Load GPT-2 language model
model = GPT2LMHeadModel.from_pretrained("gpt2")


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

STEP 6: Fix Padding Issue in GPT-2 (Very Important)

In [7]:
# GPT-2 does not have a padding token by default
# We set padding token same as end-of-sequence token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id


STEP 7: Tokenize the Dataset

In [8]:
# Convert text stories into token IDs
# Truncation limits story length for memory efficiency
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

# Apply tokenization to the entire dataset
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)


Map:   0%|          | 0/272600 [00:00<?, ? examples/s]

STEP 8: Data Collator for Language Modeling

In [9]:
# Data collator automatically creates labels from input_ids
# mlm=False because GPT-2 is a causal (next-word) language model
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)


STEP 9: Define Training Configuration

In [10]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False   # IMPORTANT: GPT-2 is NOT masked LM
)


STEP 10: Fine-Tune GPT-2 Model

In [16]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
model.to(device)


Using device: cuda


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [24]:
# Trainer handles training loop, loss calculation, and optimization
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# Start fine-tuning (limited steps → fast training)
trainer.train()

# Save the fine-tuned model
trainer.save_model("./gpt2_story_model")

Step,Training Loss
50,2.588159
100,2.769023
150,2.780482
200,3.005118
250,3.150627
300,3.283030
350,3.341344
400,3.384217
450,3.459351
500,3.530426


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

STEP 11: Generate Stories Using Fine-Tuned Model

In [25]:
from transformers import pipeline

# Create text generation pipeline
generator = pipeline(
    "text-generation",
    model="./gpt2_story_model",
    tokenizer=tokenizer
)

# Provide a story prompt
prompt = "In a forgotten kingdom where magic was forbidden,"

# Generate a creative story
output = generator(
    prompt,
    max_length=300,
    temperature=0.85,          # Controls creativity
    top_p=0.9,                 # Controls randomness
    repetition_penalty=1.1,    # Reduces repetition
    do_sample=True
)

print(output[0]["generated_text"])


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_length', 'top_p', 'repetition_penalty', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=300) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In a forgotten kingdom where magic was forbidden, the man who had given it to us is now called King of Souls. 
 It has been three years since he lost his home and all that remained were our old friends - my brother-in law's daughter as well like some cousins in their late 20s -- but I can hear them crying for help from behind me on this hill outside Haddonfield ( we only just got out after hearing they are hungry ). In those moments when no one else seems about: The sun goes down with everyone taking care not see anything too big or large; even if what could be there hurts you so hard its easy being caught by an angry monster... There will always have gotten up until soon before midnight because everything around him feels almost real.'' This word would never go away any more than every other words used today through normal channels put into place across town callers' throats anyway! We've lived here long enough though yet still nothing keeps anyone company at night anyways without som

In [26]:
prompt = "In a forgotten kingdom where magic was forbidden"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.75,
    top_p=0.9,
    repetition_penalty=1.3,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(output[0], skip_special_tokens=True))


In a forgotten kingdom where magic was forbidden and the elephants had become too dangerous to use, some people found themselves in an old 
 forest. They were not one of their own but they still seemed unable with this new world it has grown so large that even those who knew them could n't help feeling like� mowing ground for all sake now... ( you know there is no point talking about how much room these beings have around if what YOU challeng yourself are something else ). `` I love your son,'' said Emma' he smiled back at him as she tried just enough hard hitting into his chest which drove her against my forehead causing
